# 3D Reconstruction with gaussian Splatting --Step by Step

In [5]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt

for parent in [Path.cwd(), *Path.cwd().parents]:
    if (parent / "src" / "science_jubilee").is_dir():
        REPO_ROOT = parent
        break
else:
    raise RuntimeError("Could not locate the science_jubilee repository root")

SRC_ROOT = REPO_ROOT / "src"
for path in (SRC_ROOT, REPO_ROOT):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from science_jubilee.Vision.GS_Reconstruction.ingredients.colmap import run_colmap
from science_jubilee.Vision.GS_Reconstruction.ingredients.pre_process import run_filter_scene
from science_jubilee.Vision.GS_Reconstruction.ingredients.reconstruction import run_reconstruction
from science_jubilee.Vision.GS_Reconstruction.ingredients.post_process import run_filter_plants
from science_jubilee.Vision.GS_Reconstruction.ingredients.scaling import run_scale_by_cameras
from science_jubilee.Vision.GS_Reconstruction.ingredients.meshing import run_meshing
from science_jubilee.scripts.ingredients.snake_scan import run_scan


hardware = False

## 1 - Configure the dataset and scan

The scan captures a serpentine grid of images. Image names contain the capture index and the physical X/Y/Z position. Set `run_capture=True` only when the Jubilee hardware is connected; otherwise the pipeline reuses images already present in `images_dir`.

In [8]:
dataset_name = "Virtual_montserra_filt" if not hardware else "Latest_reconstruction"
start = [110.0, 80.0, 280.0]
stop = [250.0, 200.0, 220.0]
steps = [5, 5, 4]
delay = 2.0
iterations = 6000
run_capture = hardware

dataset_path = REPO_ROOT / "src/science_jubilee/Vision/GS_Reconstruction/Datasets" / dataset_name
images_dir = dataset_path / "input"
output_path = REPO_ROOT / "src/science_jubilee/Vision/GS_Reconstruction/Outputs" / f"{dataset_name}_results"
output_reconstruction = output_path / "GS_reconstruction"
images_dir.mkdir(parents=True, exist_ok=True)
output_path.mkdir(parents=True, exist_ok=True)

## 2 - Capture the snake scan

`run_scan` moves through the configured X/Y grid, captures one image at each position, and saves coordinate-aware names in the dataset input folder. With `run_capture=False`, this cell verifies that input images already exist instead of moving the machine.

In [36]:
if run_capture:
    saved_images = run_scan(
        start=start,
        stop=stop,
        steps=steps,
        delay=delay,
        out=str(images_dir),
    )
else:
    saved_images = sorted(str(path) for path in images_dir.glob("*.jpg"))
    if not saved_images:
        raise FileNotFoundError(f"No input images found in {images_dir}")

print(f"Using {len(saved_images)} images from {images_dir}")

Using 100 images from c:\Users\Justin\Desktop\Jubilee\science_jubilee\src\science_jubilee\Vision\GS_Reconstruction\Datasets\Virtual_montserra_2\input


## 3 - Prepare COLMAP camera data

COLMAP detects image features and estimates camera poses from the captured input images. The ingredient runs the repository's WSL helper script and creates the dataset's `images` and camera metadata.

In [10]:
colmap_script = REPO_ROOT / "src/science_jubilee/Vision/GS_Reconstruction/src/run_colmap.sh"
if not colmap_script.exists():
    raise FileNotFoundError(f"COLMAP script not found: {colmap_script}")

run_colmap(
    colmap_script=str(colmap_script),
    dataset_path=dataset_path,
)

'Colmap succed'

## 4 - Filter the reconstruction scene

The preprocessing step removes the tray/background from the COLMAP image set. It prepares the scene images used by Gaussian Splatting and can use the repository's AI-based filtering.

In [16]:
run_filter_scene(
    images_path=dataset_path / "images",
    use_ai=True,
)

Loading weights:   0%|          | 0/754 [00:00<?, ?it/s]

Processing c:\Users\Justin\Desktop\Jubilee\science_jubilee\src\science_jubilee\Vision\GS_Reconstruction\Datasets\virtual_basilic_2\images\img_x110_y110_z220.jpg...
Aucun code ArUco détecté.
Sauvegardé avec transparence : c:\Users\Justin\Desktop\Jubilee\science_jubilee\src\science_jubilee\Vision\GS_Reconstruction\Datasets\virtual_basilic_2\images\img_x110_y110_z220.png
Fichier original supprimé : c:\Users\Justin\Desktop\Jubilee\science_jubilee\src\science_jubilee\Vision\GS_Reconstruction\Datasets\virtual_basilic_2\images\img_x110_y110_z220.jpg
Processing c:\Users\Justin\Desktop\Jubilee\science_jubilee\src\science_jubilee\Vision\GS_Reconstruction\Datasets\virtual_basilic_2\images\img_x110_y110_z245.jpg...
Aucun code ArUco détecté.
Sauvegardé avec transparence : c:\Users\Justin\Desktop\Jubilee\science_jubilee\src\science_jubilee\Vision\GS_Reconstruction\Datasets\virtual_basilic_2\images\img_x110_y110_z245.png
Fichier original supprimé : c:\Users\Justin\Desktop\Jubilee\science_jubilee\src\

True

## 5 - Train the Gaussian Splatting model

This stage optimizes the Gaussian scene representation from the COLMAP cameras and filtered images. The trained point cloud is written under the selected iteration directory.

In [17]:
reconstruction_script = REPO_ROOT / "src/science_jubilee/Vision/GS_Reconstruction/src/run_reconstruction.sh"
if not reconstruction_script.exists():
    raise FileNotFoundError(f"Reconstruction script not found: {reconstruction_script}")

run_reconstruction(
    dataset_path=dataset_path,
    output_path=output_reconstruction,
    iterations=iterations,
    reconstruction_script=str(reconstruction_script),
)


True

In [18]:
input_ply = output_reconstruction / "point_cloud" / f"iteration_{iterations}" / "point_cloud.ply"
print(input_ply)

c:\Users\Justin\Desktop\Jubilee\science_jubilee\src\science_jubilee\Vision\GS_Reconstruction\Outputs\virtual_basilic_2_results\GS_reconstruction\point_cloud\iteration_7000\point_cloud.ply


## 6 - Remove non-plant Gaussians

The post-processing ingredient filters the trained point cloud using geometric, opacity, and color thresholds. The result is saved as the input for camera-based scaling.

In [19]:
filtered_ply = output_reconstruction / "point_cloud" / "iteration_35000" / "point_cloud.ply"
run_filter_plants(
    input_ply=input_ply,
    output_ply=filtered_ply,
        bbox_size=10000,
        bbox_center=[0.0, 2, 0.0],
        elongation_threshold=7.0,
        scale_threshold=1,
        std_ratio=3,
        opacity_threshold=0.05,
        nb_neighbors=60,
        white_sat_thresh=0.15,
        white_val_thresh=0.12,
)
print(filtered_ply)

c:\Users\Justin\Desktop\Jubilee\science_jubilee\src\science_jubilee\Vision\GS_Reconstruction\Outputs\virtual_basilic_2_results\GS_reconstruction\point_cloud\iteration_35000\point_cloud.ply


## 7 - Scale and align the point cloud

Camera poses and the known scan geometry provide the scale and orientation of the filtered point cloud. This creates a scaled PLY file using the reconstruction camera metadata.

In [ ]:
scaled_ply = output_reconstruction / "point_cloud" / "iteration_35000" / "point_cloud_scaled.ply"
run_scale_by_cameras(
    input_ply=filtered_ply,
    output_ply=scaled_ply,
    cameras_json_path=output_reconstruction / "cameras.json",
    cameras_span=None,
    rot=[0,0,0]
)
print(scaled_ply)

c:\Users\Justin\Desktop\Jubilee\science_jubilee\src\science_jubilee\Vision\GS_Reconstruction\Outputs\virtual_basilic_2_results\GS_reconstruction\point_cloud\iteration_35000\point_cloud_scaled.ply


In [ ]:
input_colmap= dataset_path/"dense_point_cloud.ply"
scaled_colmap = output_path/ "Dense_scaled.ply"
run_scale_by_cameras(
    input_ply=input_colmap,
    output_ply=scaled_colmap,
    cameras_json_path=output_reconstruction / "cameras.json",
    cameras_span=None,
    rot=[0,0,0]
)
print(scaled_colmap)

NameError: name 'scaled_ply' is not defined

## 8 - Build the mesh

The final ingredient converts the scaled Gaussian point cloud into an OBJ mesh using an alpha shape and optional decimation. The mesh is written beside the reconstruction output.

In [22]:
mesh_path = output_path / "mesh.obj"
run_meshing(
    input_ply=scaled_ply,
    output_obj=mesh_path,
    alpha=0.0038,
    decimate_ratio=0.8
)
print(mesh_path)

[Open3D WARNING] Write OBJ can not include triangle normals.
c:\Users\Justin\Desktop\Jubilee\science_jubilee\src\science_jubilee\Vision\GS_Reconstruction\Outputs\virtual_basilic_2_results\mesh.obj


## Visualisers

SIBR visualiser for gaussian splattings

In [11]:
import os
Viewer_path = (
            REPO_ROOT / "src/science_jubilee/Vision/3D_Reconstruction/Viewer/bin"
        )
        # Gaussian Viewer
os.system(
            f"cd {Viewer_path} && SIBR_gaussianViewer_app.exe -m {output_reconstruction }"
        )

1

Open3d visualiser for the created 3d mesh

In [23]:
import open3d as o3d
# Display the mesh
mesh = o3d.io.read_triangle_mesh(str(mesh_path))
o3d.visualization.draw_geometries([mesh], mesh_show_back_face=True)

# Features extracting

Thanks to themeshes and only using the point cloud of the reconstuction(center of the gaussians) we are able to extract information about our plant.

For ewample we have created a script to extract the leafs and find the horizontal ones

In [19]:
import itertools
import open3d as o3d

from science_jubilee.Vision.GS_Reconstruction.ingredients.extract_leafs import (
    run_extract_leaf_clusters,
)


mesh = o3d.io.read_triangle_mesh(str(mesh_path))
if not mesh.has_vertices():
    raise ValueError(f"Mesh has no vertices: {mesh_path}")

mesh_pcd = o3d.geometry.PointCloud(mesh.vertices)
parameter_grid = {
    "distance_threshold": [0.00001, 0.0001, 0.0005, 0.001, 0.003, 0.01, 0.03, 0.1],
    "min_points": [2, 3, 5, 10, 20, 50],
    "size_threshold": [0.00001, 0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05, 0.1, 0.2],
    "shape_threshold": [0.20, 0.30, 0.40, 0.50, 0.65, 0.80, 0.95, 0.99],
    "height_ratio": [0.0, 0.1, 0.2, 0.4, 0.6, 0.8],
}
target_clusters = 15
search_results = []
total_combinations = 1
for values in parameter_grid.values():
    total_combinations *= len(values)

for combination_index, values in enumerate(
    itertools.product(*parameter_grid.values()), start=1
):
    parameters = dict(zip(parameter_grid, values))
    clusters = run_extract_leaf_clusters(pcd=mesh_pcd, **parameters)
    cluster_count = len(clusters)
    search_results.append(
        {
            "error": abs(cluster_count - target_clusters),
            "cluster_count": cluster_count,
            "parameters": parameters,
        }
    )
    if combination_index % 1000 == 0:
        print(f"Tested {combination_index}/{total_combinations} combinations")

search_results.sort(key=lambda result: (result["error"], -result["cluster_count"]))
best_result = search_results[0]
best_parameters = best_result["parameters"]
leaf_clusters = run_extract_leaf_clusters(pcd=mesh_pcd, **best_parameters)

print(f"Tested {total_combinations} combinations")
print(f"Best result: {best_result['cluster_count']} clusters")
print(f"Best parameters: {best_parameters}")
print("Top 10 candidates:")
for result in search_results[:10]:
    print(result["cluster_count"], result["parameters"])


Tested 1000/20736 combinations
Tested 2000/20736 combinations
Tested 3000/20736 combinations


KeyboardInterrupt: 

In [24]:
import numpy as np
import open3d as o3d

from science_jubilee.Vision.GS_Reconstruction.ingredients.extract_leafs import (
    run_compute_leaf_normals,
    run_extract_normal_leafs,
    run_extract_leaf_clusters,
)


mesh = o3d.io.read_triangle_mesh(str(mesh_path))
if not mesh.has_vertices():
    raise ValueError(f"Mesh has no vertices: {mesh_path}")

mesh_pcd = o3d.geometry.PointCloud(mesh.vertices)
leaf_clusters = run_extract_leaf_clusters(
    pcd=mesh_pcd,
    distance_threshold=0.0086,  #use 0.0092 to get the full leafs and 0.086 to get different leafs zones
    min_points=20,
    size_threshold=1e-5,
    shape_threshold=0.98,
    height_ratio=0.1,
)

horizontal_threshold = 0.90
normals = run_compute_leaf_normals(leaf_clusters=leaf_clusters)
plant_z = np.array([0.0, 1.0, 0.0])
normal_z_dots = [
    np.nan if np.isscalar(normal) else abs(np.dot(normal, plant_z))
    for normal in normals
]
horizontal_leaf_clusters = run_extract_normal_leafs(
    leaf_clusters=leaf_clusters,
    horizontal_threshold=horizontal_threshold,
)
horizontal_leaf_ids = {id(leaf) for leaf in horizontal_leaf_clusters}

print("Normal dot Z for each leaf:")
for index, normal_z_dot in enumerate(normal_z_dots, start=1):
    print(f"Leaf {index}: {normal_z_dot:.4f}")
print(
    f"Horizontal leaves: {len(horizontal_leaf_clusters)}/{len(leaf_clusters)} "
    f"(|dot| >= {horizontal_threshold})"
)

geometries = [mesh]
#mesh.paint_uniform_color([0.65, 0.65, 0.65])

for index, leaf in enumerate(leaf_clusters, start=1):
    is_horizontal = id(leaf) in horizontal_leaf_ids
    leaf.paint_uniform_color([0.1, 0.8, 0.2] if is_horizontal else [0.7, 0.7, 0.7])
    bounding_box = leaf.get_axis_aligned_bounding_box()
    bounding_box.color = [1.0, 0.1, 0.1] if is_horizontal else [0.3, 0.3, 0.3]
    geometries.extend([leaf, bounding_box])

    normal = normals[index - 1]
    if np.isscalar(normal) or np.any(np.isnan(normal)):
        continue
    points = np.asarray(leaf.points)
    centroid = points.mean(axis=0)
    leaf_extent = np.linalg.norm(bounding_box.get_extent())
    arrow_length = max(leaf_extent * 0.35, 0.005)
    arrow = o3d.geometry.TriangleMesh.create_arrow(
        cylinder_radius=arrow_length * 0.025,
        cone_radius=arrow_length * 0.08,
        cylinder_height=arrow_length * 0.75,
        cone_height=arrow_length * 0.25,
    )

    z_axis = np.array([0.0, 0.0, 1.0])
    rotation_vector = np.cross(z_axis, normal)
    rotation_vector_norm = np.linalg.norm(rotation_vector)
    rotation_dot = np.clip(np.dot(z_axis, normal), -1.0, 1.0)
    if rotation_vector_norm < 1e-12:
        rotation = np.eye(3) if rotation_dot >= 0 else np.diag([1.0, -1.0, -1.0])
    else:
        skew = np.array([
            [0.0, -rotation_vector[2], rotation_vector[1]],
            [rotation_vector[2], 0.0, -rotation_vector[0]],
            [-rotation_vector[1], rotation_vector[0], 0.0],
        ])
        rotation = (
            np.eye(3)
            + skew
            + skew @ skew * ((1.0 - rotation_dot) / rotation_vector_norm**2)
        )
    arrow.rotate(rotation, center=[0.0, 0.0, 0.0])
    arrow.translate(centroid)
    arrow.paint_uniform_color([1.0, 0.55, 0.0])
    geometries.append(arrow)

print("Orange arrows show the PCA normal of each leaf")
o3d.visualization.draw_geometries(
    geometries,
    window_name="Leaf PCA normals and bounding boxes",
    point_show_normal=False,
    mesh_show_back_face=True,
)

Normal dot Z for each leaf:
Leaf 1: 0.6092
Leaf 2: 0.7801
Leaf 3: 0.8479
Leaf 4: 0.8051
Leaf 5: 0.6134
Leaf 6: 0.9697
Leaf 7: 0.9952
Leaf 8: 0.9222
Leaf 9: 0.8376
Leaf 10: 0.1837
Leaf 11: 0.0269
Leaf 12: 0.8503
Leaf 13: 0.6135
Leaf 14: 0.6225
Leaf 15: 0.4350
Leaf 16: 0.8831
Leaf 17: 0.5431
Leaf 18: 0.8550
Leaf 19: 0.8889
Leaf 20: 0.2992
Leaf 21: 0.7144
Leaf 22: 0.4333
Leaf 23: 0.7817
Leaf 24: 0.3821
Leaf 25: 0.8362
Leaf 26: 0.1303
Leaf 27: 0.4802
Leaf 28: 0.7526
Leaf 29: 0.5206
Leaf 30: 0.6081
Leaf 31: 0.8892
Leaf 32: 0.5402
Leaf 33: 0.3076
Leaf 34: 0.7850
Leaf 35: 0.6641
Leaf 36: 0.7185
Leaf 37: 0.9763
Leaf 38: 0.6672
Leaf 39: 0.6579
Leaf 40: 0.6037
Leaf 41: 0.9622
Leaf 42: 0.9702
Leaf 43: 0.3554
Leaf 44: 0.6981
Leaf 45: 0.6883
Leaf 46: 0.3183
Leaf 47: 0.6136
Leaf 48: 0.9300
Leaf 49: 0.3141
Leaf 50: 0.6987
Horizontal leaves: 7/50 (|dot| >= 0.9)
Orange arrows show the PCA normal of each leaf
